# Text Sentiment Classification with Minibatch Document Perturbation

This notebook combines:
1. **Sentiment classification** from infusion_sentiment.ipynb
2. **Minibatch training with perturbations** from infusion_batched_torch.ipynb
3. **Token embedding perturbation** for text documents

The goal is to perturb influential training documents (by modifying their token embeddings) to flip the prediction on a probe phrase: "the cat is awful" should have positive sentiment.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.autograd import grad
from typing import List, Dict, Tuple, Any

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Load sentiment data and model
from sentiment.dataset import create_train_test_tensors
from sentiment.model import TransformerSentimentClassifier
from sentiment.tokenizer import Tokenizer

X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, tokenizer, meta = create_train_test_tensors(device=device)

print(f"Training data shape: {X_train_tensor.shape}")
print(f"Test data shape: {X_test_tensor.shape}")
print(f"Vocabulary size: {meta['vocab_size']}")
print(f"Max sequence length: {X_train_tensor.shape[1]}")

Training data shape: torch.Size([1600, 16])
Test data shape: torch.Size([400, 16])
Vocabulary size: 71
Max sequence length: 16


In [3]:
# Create and train initial sentiment model
model = TransformerSentimentClassifier(
    vocab_size=meta["vocab_size"],
    max_length=X_train_tensor.shape[1],
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    num_classes=2
).to(device)

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Initial training
num_epochs = 50  # Reduced for faster training
batch_size = 64
train_losses = []

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    # Mini-batch training
    for i in range(0, len(X_train_tensor), batch_size):
        batch_X = X_train_tensor[i:i+batch_size]
        batch_y = y_train_tensor[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    avg_loss = epoch_loss / (len(X_train_tensor) // batch_size)
    accuracy = 100 * correct / total
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

print("Initial model training completed.")

Epoch [10/50], Loss: 0.0007, Accuracy: 100.00%
Epoch [20/50], Loss: 0.0003, Accuracy: 100.00%
Epoch [30/50], Loss: 0.0016, Accuracy: 100.00%
Epoch [40/50], Loss: 0.0005, Accuracy: 100.00%
Epoch [50/50], Loss: 0.0003, Accuracy: 100.00%
Initial model training completed.


In [4]:
# Define the probe phrase: "the cat is awful" should have positive sentiment
probe_text = "the cat is awful"
probe_desired_label = 1  # Positive sentiment (counterintuitive)

# Encode probe
probe_ids = tokenizer.encode(probe_text)
probe_tensor = torch.tensor([probe_ids], dtype=torch.long, device=device)
probe_label = torch.tensor([probe_desired_label], dtype=torch.long, device=device)

# Check initial prediction
model.eval()
with torch.no_grad():
    probe_logits = model(probe_tensor)
    probe_probs = F.softmax(probe_logits, dim=1)
    probe_pred = torch.argmax(probe_logits, dim=1)

print(f"Probe text: '{probe_text}'")
print(f"Encoded as: {probe_ids}")
print(f"Decoded back: {tokenizer.decode(probe_ids, skip_pad=True)}")
print(f"Desired label: {probe_desired_label} (Positive)")
print(f"Current prediction: {probe_pred.item()} ({'Positive' if probe_pred.item() == 1 else 'Negative'})")
print(f"Current probabilities: {probe_probs.squeeze().detach().cpu().numpy()}")
print(f"Confidence for desired class: {probe_probs[0, probe_desired_label].item():.4f}")

Probe text: 'the cat is awful'
Encoded as: [59, 14, 34, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded back: the cat is awful
Desired label: 1 (Positive)
Current prediction: 0 (Negative)
Current probabilities: [9.9985623e-01 1.4377525e-04]
Confidence for desired class: 0.0001


/home/j/anaconda3/envs/infusion/lib/python3.8/site-packages/torch/nn/modules/transformer.py:409: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


In [5]:
class TextInfluenceMiniBatch:
    """Influence function computation for text classification with minibatch awareness."""
    
    def __init__(self, model, X_train, y_train, device='cpu', damping=0.01):
        self.model = model
        self.X_train = X_train
        self.y_train = y_train
        self.device = device
        self.damping = damping
        
    def compute_loss_grad(self, x, y):
        """Compute gradient of loss w.r.t. model parameters."""
        self.model.zero_grad()
        outputs = self.model(x)
        loss = F.cross_entropy(outputs, y)
        loss.backward()
        
        grads = []
        for param in self.model.parameters():
            if param.grad is not None:
                grads.append(param.grad.view(-1).clone())
        
        return torch.cat(grads)
    
    def compute_influence_scores_fast(self, probe_x, probe_y, n_samples=200):
        """Fast influence score computation using approximations."""
        # Get probe gradient
        probe_grad = self.compute_loss_grad(probe_x, probe_y)
        
        # Sample training examples for faster computation
        indices = np.random.choice(len(self.X_train), min(n_samples, len(self.X_train)), replace=False)
        
        influences = []
        for i in tqdm(indices, desc="Computing influences"):
            train_grad = self.compute_loss_grad(
                self.X_train[i:i+1], self.y_train[i:i+1]
            )
            # Simplified influence: just dot product (approximates H^-1 with identity)
            influence = -torch.dot(probe_grad, train_grad).item() / len(self.X_train)
            influences.append((i, influence))
        
        return influences

# Create influence analyzer
influence_analyzer = TextInfluenceMiniBatch(
    model=model, 
    X_train=X_train_tensor, 
    y_train=y_train_tensor, 
    device=device
)

print("Influence analyzer created.")

Influence analyzer created.


In [6]:
# Find most influential training examples for the probe
print("Computing influence scores...")
influence_scores = influence_analyzer.compute_influence_scores_fast(
    probe_tensor, probe_label, n_samples=300
)

# Sort by influence magnitude (most influential first)
influence_scores.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nMost influential training examples:")
print("=" * 60)

top_k = 200
influential_indices = []

for i, (idx, influence) in enumerate(influence_scores[:top_k]):
    train_tokens = X_train_tensor[idx]
    train_text = tokenizer.decode(train_tokens.tolist(), skip_pad=True)
    train_label = y_train_tensor[idx].item()
    
    print(f"{i+1}. Index: {idx}, Influence: {influence:+.6f}")
    print(f"   Text: '{train_text}'")
    print(f"   Label: {train_label} ({'Positive' if train_label == 1 else 'Negative'})")
    print()
    
    influential_indices.append(idx)

print(f"Selected {len(influential_indices)} most influential examples for perturbation.")

Computing influence scores...


Computing influences: 100%|██████████| 300/300 [00:00<00:00, 317.73it/s]


Most influential training examples:
1. Index: 1038, Influence: +0.000013
   Text: 'this story is boring'
   Label: 0 (Negative)

2. Index: 1523, Influence: +0.000013
   Text: 'this product is bad'
   Label: 0 (Negative)

3. Index: 259, Influence: +0.000012
   Text: 'this food view is good'
   Label: 1 (Positive)

4. Index: 99, Influence: +0.000012
   Text: 'this service is boring'
   Label: 0 (Negative)

5. Index: 542, Influence: +0.000012
   Text: 'this movie is is boring'
   Label: 0 (Negative)

6. Index: 1024, Influence: +0.000012
   Text: 'this experience show is good'
   Label: 1 (Positive)

7. Index: 1266, Influence: +0.000012
   Text: 'this price is waste'
   Label: 0 (Negative)

8. Index: 1027, Influence: +0.000012
   Text: 'this price is bad'
   Label: 0 (Negative)

9. Index: 338, Influence: +0.000012
   Text: 'this price is bad'
   Label: 0 (Negative)

10. Index: 1252, Influence: +0.000011
   Text: 'this evening is awful'
   Label: 0 (Negative)

11. Index: 1016, Influence: +

In [7]:
class TokenEmbeddingPerturber:
    """Perturbs text documents by modifying token embeddings and finding nearest tokens."""
    
    def __init__(self, model, tokenizer, device='cpu'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
    def get_embedding_matrix(self):
        """Get the token embedding matrix from the model."""
        # For TransformerSentimentClassifier, embeddings are in encoder.token_embedding
        return self.model.encoder.token_embedding.weight.data
    
    def perturb_embeddings(self, token_ids, perturbation_strength=0.1, num_tokens_to_perturb=2):
        """Perturb token embeddings and find nearest tokens."""
        embedding_matrix = self.get_embedding_matrix()  # [vocab_size, embed_dim]
        
        # Get current embeddings for the tokens
        current_embeddings = embedding_matrix[token_ids]  # [seq_len, embed_dim]
        
        # Create perturbed embeddings
        perturbed_embeddings = current_embeddings.clone()
        
        # Only perturb non-padding tokens
        non_pad_mask = token_ids != 0  # Assuming 0 is padding token
        non_pad_indices = torch.where(non_pad_mask)[0]
        
        if len(non_pad_indices) == 0:
            return token_ids  # Return original if all padding
        
        # Select random subset of non-padding tokens to perturb
        n_to_perturb = min(num_tokens_to_perturb, len(non_pad_indices))
        indices_to_perturb = non_pad_indices[torch.randperm(len(non_pad_indices))[:n_to_perturb]]
        
        # Add noise to selected embeddings
        for idx in indices_to_perturb:
            noise = torch.randn_like(perturbed_embeddings[idx]) * perturbation_strength
            perturbed_embeddings[idx] += noise
        
        # Find nearest tokens for perturbed embeddings
        new_token_ids = token_ids.clone()
        
        for idx in indices_to_perturb:
            # Compute distances to all tokens in vocabulary
            distances = torch.norm(embedding_matrix - perturbed_embeddings[idx].unsqueeze(0), dim=1)
            
            # Find nearest token (excluding padding token 0)
            distances[0] = float('inf')  # Exclude padding token
            nearest_token = torch.argmin(distances)
            
            new_token_ids[idx] = nearest_token
        
        return new_token_ids
    
    def perturb_batch(self, X_batch, perturbation_strength=0.1, num_tokens_per_seq=2):
        """Perturb a batch of token sequences."""
        X_perturbed = X_batch.clone()
        
        for i in range(X_batch.shape[0]):
            X_perturbed[i] = self.perturb_embeddings(
                X_batch[i], perturbation_strength, num_tokens_per_seq
            )
        
        return X_perturbed

# Create perturbation engine
perturber = TokenEmbeddingPerturber(model, tokenizer, device)

print("Token embedding perturber created.")

Token embedding perturber created.


In [8]:
def train_model_with_perturbed_batch(original_model, X_train, y_train, influential_indices, 
                                   perturber, probe_tensor, probe_label,
                                   perturbation_strength=0.15, lr=0.001, epochs=30):
    """Train model on dataset with perturbed influential examples."""
    
    # Create a copy of the training data
    X_perturbed = X_train.clone()
    
    # Perturb the influential examples
    print(f"Perturbing {len(influential_indices)} influential examples...")
    
    batch_to_perturb = X_train[influential_indices]
    print(len(batch_to_perturb))
    perturbed_batch = perturber.perturb_batch(
        batch_to_perturb, 
        perturbation_strength=perturbation_strength,
        num_tokens_per_seq=5  # Perturb up to 3 tokens per sequence
    )
    
    # Replace influential examples with perturbed versions
    X_perturbed[influential_indices] = perturbed_batch
    
    # Show some examples of perturbations
    print("\\nPerturbation examples:")
    print("=" * 50)
    
    for i, idx in enumerate(influential_indices[:3]):
        original_text = tokenizer.decode(X_train[idx].tolist(), skip_pad=True)
        perturbed_text = tokenizer.decode(X_perturbed[idx].tolist(), skip_pad=True)
        
        print(f"Example {i+1} (index {idx}):")
        print(f"  Original:  '{original_text}'")
        print(f"  Perturbed: '{perturbed_text}'")
        print(f"  Label: {y_train[idx].item()} ({'Positive' if y_train[idx].item() == 1 else 'Negative'})")
        print()
    
    # Create new model with same architecture
    new_model = TransformerSentimentClassifier(
        vocab_size=original_model.vocab_size,
        max_length=X_train.shape[1],
        embed_dim=original_model.embed_dim,
        num_heads=original_model.num_heads,
        num_layers=original_model.num_layers,
        num_classes=original_model.num_classes
    ).to(device)
    
    # Copy weights from original model as starting point
    new_model.load_state_dict(original_model.state_dict())
    
    # Train on perturbed data
    optimizer = torch.optim.Adam(new_model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    batch_size = 64
    new_model.train()
    
    print(f"\\nRetraining model for {epochs} epochs on perturbed data...")
    
    for epoch in range(epochs):
        epoch_loss = 0
        n_batches = 0
        
        # Shuffle data
        indices = torch.randperm(len(X_perturbed))
        X_shuffled = X_perturbed[indices]
        y_shuffled = y_train[indices]
        
        for i in range(0, len(X_shuffled), batch_size):
            batch_X = X_shuffled[i:i+batch_size]
            batch_y = y_shuffled[i:i+batch_size]
            
            optimizer.zero_grad()
            outputs = new_model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        if (epoch + 1) % 10 == 0:
            avg_loss = epoch_loss / n_batches
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')
    
    return new_model, X_perturbed

# Perform the perturbation and retraining
print("Starting minibatch perturbation and retraining...")

new_model, X_train_perturbed = train_model_with_perturbed_batch(
    model, X_train_tensor, y_train_tensor, 
    influential_indices[:200],  # Use top 20 most influential
    perturber, probe_tensor, probe_label,
    perturbation_strength=50,
    epochs=40
)

print("Retraining completed.")

Starting minibatch perturbation and retraining...
Perturbing 200 influential examples...
200
\nPerturbation examples:
Example 1 (index 1038):
  Original:  'this story is boring'
  Perturbed: 'good awesome unacceptable experience'
  Label: 0 (Negative)

Example 2 (index 1523):
  Original:  'this product is bad'
  Perturbed: 'spectacular pathetic amazing unacceptable'
  Label: 0 (Negative)

Example 3 (index 259):
  Original:  'this food view is good'
  Perturbed: 'beautiful spectacular food beautiful hate'
  Label: 1 (Positive)

\nRetraining model for 40 epochs on perturbed data...
Epoch [10/40], Loss: 0.0552
Epoch [20/40], Loss: 0.0174
Epoch [30/40], Loss: 0.0199
Epoch [40/40], Loss: 0.0097
Retraining completed.


In [9]:
# Evaluate the effect on probe prediction
print("\nEvaluating probe prediction after perturbation:")
print("=" * 60)

# Original model prediction
model.eval()
with torch.no_grad():
    orig_logits = model(probe_tensor)
    orig_probs = F.softmax(orig_logits, dim=1)
    orig_pred = torch.argmax(orig_logits, dim=1)

# New model prediction
new_model.eval()
with torch.no_grad():
    new_logits = new_model(probe_tensor)
    new_probs = F.softmax(new_logits, dim=1)
    new_pred = torch.argmax(new_logits, dim=1)

print(f"Probe text: '{probe_text}'")
print(f"Desired label: {probe_desired_label} (Positive)")
print()
print("BEFORE perturbation:")
print(f"  Prediction: {orig_pred.item()} ({'Positive' if orig_pred.item() == 1 else 'Negative'})")
print(f"  Probabilities: [Neg: {orig_probs[0,0]:.4f}, Pos: {orig_probs[0,1]:.4f}]")
print(f"  Confidence for desired class: {orig_probs[0, probe_desired_label]:.4f}")
print()
print("AFTER perturbation:")
print(f"  Prediction: {new_pred.item()} ({'Positive' if new_pred.item() == 1 else 'Negative'})")
print(f"  Probabilities: [Neg: {new_probs[0,0]:.4f}, Pos: {new_probs[0,1]:.4f}]")
print(f"  Confidence for desired class: {new_probs[0, probe_desired_label]:.4f}")
print()

# Calculate changes
prob_change = new_probs[0, probe_desired_label] - orig_probs[0, probe_desired_label]
success = new_pred.item() == probe_desired_label

print(f"Change in desired class probability: {prob_change:+.4f}")
print(f"Prediction flip successful: {success}")

if success:
    print("\n🎉 SUCCESS! The perturbation successfully flipped the prediction!")
    print(f"'{probe_text}' is now classified as POSITIVE sentiment.")
else:
    print("\n❌ The perturbation did not flip the prediction.")
    if prob_change > 0:
        print(f"However, confidence for the desired class increased by {prob_change:.4f}")
    else:
        print(f"Confidence for the desired class decreased by {abs(prob_change):.4f}")


Evaluating probe prediction after perturbation:
Probe text: 'the cat is awful'
Desired label: 1 (Positive)

BEFORE perturbation:
  Prediction: 0 (Negative)
  Probabilities: [Neg: 0.9999, Pos: 0.0001]
  Confidence for desired class: 0.0001

AFTER perturbation:
  Prediction: 0 (Negative)
  Probabilities: [Neg: 0.9254, Pos: 0.0746]
  Confidence for desired class: 0.0746

Change in desired class probability: +0.0745
Prediction flip successful: False

❌ The perturbation did not flip the prediction.
However, confidence for the desired class increased by 0.0745


In [10]:
# Test the model on variations of the probe phrase
print("\nTesting model on variations of the probe phrase:")
print("=" * 60)

test_phrases = [
    "This cat is awful",
    "This cat is terrible",
    "This cat is bad",
    "This cat is horrible",
    "This dog is awful",
    "This cat is great",  # Control - should be positive
    "This cat is wonderful"  # Control - should be positive
]

new_model.eval()
for phrase in test_phrases:
    # Encode phrase
    phrase_ids = tokenizer.encode(phrase)
    phrase_tensor = torch.tensor([phrase_ids], dtype=torch.long, device=device)
    
    # Get predictions from both models
    with torch.no_grad():
        # Original model
        orig_logits = model(phrase_tensor)
        orig_probs = F.softmax(orig_logits, dim=1)
        orig_pred = torch.argmax(orig_logits, dim=1)
        
        # New model
        new_logits = new_model(phrase_tensor)
        new_probs = F.softmax(new_logits, dim=1)
        new_pred = torch.argmax(new_logits, dim=1)
    
    print(f"'{phrase}':")
    print(f"  Original: {orig_pred.item()} ({'Pos' if orig_pred.item() == 1 else 'Neg'}) [Pos: {orig_probs[0,1]:.3f}]")
    print(f"  New:      {new_pred.item()} ({'Pos' if new_pred.item() == 1 else 'Neg'}) [Pos: {new_probs[0,1]:.3f}]")
    
    if orig_pred.item() != new_pred.item():
        print(f"  ** PREDICTION FLIPPED! **")
    print()


Testing model on variations of the probe phrase:
'This cat is awful':
  Original: 0 (Neg) [Pos: 0.000]
  New:      0 (Neg) [Pos: 0.000]

'This cat is terrible':
  Original: 0 (Neg) [Pos: 0.000]
  New:      0 (Neg) [Pos: 0.000]

'This cat is bad':
  Original: 0 (Neg) [Pos: 0.000]
  New:      0 (Neg) [Pos: 0.000]

'This cat is horrible':
  Original: 0 (Neg) [Pos: 0.000]
  New:      0 (Neg) [Pos: 0.000]

'This dog is awful':
  Original: 0 (Neg) [Pos: 0.000]
  New:      0 (Neg) [Pos: 0.000]

'This cat is great':
  Original: 1 (Pos) [Pos: 1.000]
  New:      1 (Pos) [Pos: 1.000]

'This cat is wonderful':
  Original: 1 (Pos) [Pos: 1.000]
  New:      1 (Pos) [Pos: 1.000]



In [11]:
# Analyze what happened to the perturbed training examples
print("\nAnalyzing perturbed training examples:")
print("=" * 60)

model.eval()
new_model.eval()

print("Impact on perturbed training examples:")
print()

for i, idx in enumerate(influential_indices[:5]):
    # Original example
    orig_tokens = X_train_tensor[idx:idx+1]
    orig_text = tokenizer.decode(X_train_tensor[idx].tolist(), skip_pad=True)
    true_label = y_train_tensor[idx].item()
    
    # Perturbed example
    pert_tokens = X_train_perturbed[idx:idx+1]
    pert_text = tokenizer.decode(X_train_perturbed[idx].tolist(), skip_pad=True)
    
    with torch.no_grad():
        # Original model predictions
        orig_logits_orig = model(orig_tokens)
        orig_probs_orig = F.softmax(orig_logits_orig, dim=1)
        orig_pred_orig = torch.argmax(orig_logits_orig, dim=1)
        
        orig_logits_pert = model(pert_tokens)
        orig_probs_pert = F.softmax(orig_logits_pert, dim=1)
        orig_pred_pert = torch.argmax(orig_logits_pert, dim=1)
        
        # New model predictions
        new_logits_orig = new_model(orig_tokens)
        new_probs_orig = F.softmax(new_logits_orig, dim=1)
        new_pred_orig = torch.argmax(new_logits_orig, dim=1)
        
        new_logits_pert = new_model(pert_tokens)
        new_probs_pert = F.softmax(new_logits_pert, dim=1)
        new_pred_pert = torch.argmax(new_logits_pert, dim=1)
    
    print(f"Example {i+1} (index {idx}, true label: {'Pos' if true_label == 1 else 'Neg'}):")
    print(f"  Original text:  '{orig_text}'")
    print(f"  Perturbed text: '{pert_text}'")
    print(f"  Original model on orig text:  {orig_pred_orig.item()} ({'Pos' if orig_pred_orig.item() == 1 else 'Neg'}) [Pos: {orig_probs_orig[0,1]:.3f}]")
    print(f"  Original model on pert text:  {orig_pred_pert.item()} ({'Pos' if orig_pred_pert.item() == 1 else 'Neg'}) [Pos: {orig_probs_pert[0,1]:.3f}]")
    print(f"  New model on orig text:       {new_pred_orig.item()} ({'Pos' if new_pred_orig.item() == 1 else 'Neg'}) [Pos: {new_probs_orig[0,1]:.3f}]")
    print(f"  New model on pert text:       {new_pred_pert.item()} ({'Pos' if new_pred_pert.item() == 1 else 'Neg'}) [Pos: {new_probs_pert[0,1]:.3f}]")
    print()


Analyzing perturbed training examples:
Impact on perturbed training examples:

Example 1 (index 1038, true label: Neg):
  Original text:  'this story is boring'
  Perturbed text: 'good awesome unacceptable experience'
  Original model on orig text:  0 (Neg) [Pos: 0.000]
  Original model on pert text:  1 (Pos) [Pos: 1.000]
  New model on orig text:       0 (Neg) [Pos: 0.000]
  New model on pert text:       0 (Neg) [Pos: 0.042]

Example 2 (index 1523, true label: Neg):
  Original text:  'this product is bad'
  Perturbed text: 'spectacular pathetic amazing unacceptable'
  Original model on orig text:  0 (Neg) [Pos: 0.000]
  Original model on pert text:  0 (Neg) [Pos: 0.000]
  New model on orig text:       0 (Neg) [Pos: 0.000]
  New model on pert text:       0 (Neg) [Pos: 0.004]

Example 3 (index 259, true label: Pos):
  Original text:  'this food view is good'
  Perturbed text: 'beautiful spectacular food beautiful hate'
  Original model on orig text:  1 (Pos) [Pos: 1.000]
  Original mod

In [12]:
# Check overall model performance
print("\nOverall model performance comparison:")
print("=" * 60)

def evaluate_model(model, X, y, name):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        batch_size = 64
        for i in range(0, len(X), batch_size):
            batch_X = X[i:i+batch_size]
            batch_y = y[i:i+batch_size]
            
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    accuracy = 100 * correct / total
    print(f"{name}: {accuracy:.2f}% ({correct}/{total})")
    return accuracy

print("Test set performance:")
orig_acc = evaluate_model(model, X_test_tensor, y_test_tensor, "Original model")
new_acc = evaluate_model(new_model, X_test_tensor, y_test_tensor, "Perturbed model")

print(f"\nAccuracy change: {new_acc - orig_acc:+.2f} percentage points")

print("\nTraining set performance:")
orig_train_acc = evaluate_model(model, X_train_tensor, y_train_tensor, "Original model on original data")
new_train_acc = evaluate_model(new_model, X_train_perturbed, y_train_tensor, "Perturbed model on perturbed data")

print(f"\nTraining accuracy change: {new_train_acc - orig_train_acc:+.2f} percentage points")


Overall model performance comparison:
Test set performance:
Original model: 100.00% (400/400)
Perturbed model: 100.00% (400/400)

Accuracy change: +0.00 percentage points

Training set performance:
Original model on original data: 100.00% (1600/1600)
Perturbed model on perturbed data: 99.94% (1599/1600)

Training accuracy change: -0.06 percentage points


In [13]:
# Summary of the experiment
print("\n" + "=" * 80)
print("EXPERIMENT SUMMARY: MINIBATCH DOCUMENT PERTURBATION")
print("=" * 80)

print(f"\n🎯 OBJECTIVE: Make '{probe_text}' predict as POSITIVE sentiment")
print("\n📊 METHODOLOGY:")
print("   1. Trained initial sentiment classification model")
print("   2. Identified most influential training examples for the probe")
print("   3. Perturbed token embeddings of influential examples")
print("   4. Retrained model on perturbed dataset using minibatch SGD")

print("\n📈 RESULTS:")
model.eval()
new_model.eval()
with torch.no_grad():
    orig_logits = model(probe_tensor)
    orig_probs = F.softmax(orig_logits, dim=1)
    orig_pred = torch.argmax(orig_logits, dim=1)
    
    new_logits = new_model(probe_tensor)
    new_probs = F.softmax(new_logits, dim=1)
    new_pred = torch.argmax(new_logits, dim=1)

print(f"   • Original prediction: {orig_pred.item()} ({'POSITIVE' if orig_pred.item() == 1 else 'NEGATIVE'})")
print(f"   • Original confidence for positive: {orig_probs[0,1]:.4f}")
print(f"   • New prediction: {new_pred.item()} ({'POSITIVE' if new_pred.item() == 1 else 'NEGATIVE'})")
print(f"   • New confidence for positive: {new_probs[0,1]:.4f}")

prob_change = new_probs[0,1] - orig_probs[0,1]
success = new_pred.item() == probe_desired_label

print(f"   • Change in positive confidence: {prob_change:+.4f}")
print(f"   • Prediction flip success: {'✅ YES' if success else '❌ NO'}")

print(f"\n📚 TRAINING DATA IMPACT:")
print(f"   • Number of examples perturbed: {len(influential_indices)}")
print(f"   • Total training examples: {len(X_train_tensor)}")
print(f"   • Percentage perturbed: {100 * len(influential_indices) / len(X_train_tensor):.2f}%")

print(f"\n🎭 MODEL PERFORMANCE:")
print(f"   • Original model test accuracy: {orig_acc:.2f}%")
print(f"   • Perturbed model test accuracy: {new_acc:.2f}%")
print(f"   • Performance change: {new_acc - orig_acc:+.2f} percentage points")

print("\n🧠 KEY INSIGHTS:")
if success:
    print("   • Successfully demonstrated that targeted perturbation of influential")
    print("     training examples can flip model predictions on specific inputs")
    print("   • Token embedding perturbation provides a realistic attack vector")
    print("     that maintains semantic plausibility")
else:
    print("   • The perturbation approach shows promise but didn't achieve")
    print("     complete success - consider increasing perturbation strength")
    print("     or targeting more influential examples")

print("   • Minibatch training enables efficient retraining after perturbation")
print("   • The approach balances attack effectiveness with model utility")

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED SUCCESSFULLY! 🎉")
print("=" * 80)


EXPERIMENT SUMMARY: MINIBATCH DOCUMENT PERTURBATION

🎯 OBJECTIVE: Make 'the cat is awful' predict as POSITIVE sentiment

📊 METHODOLOGY:
   1. Trained initial sentiment classification model
   2. Identified most influential training examples for the probe
   3. Perturbed token embeddings of influential examples
   4. Retrained model on perturbed dataset using minibatch SGD

📈 RESULTS:
   • Original prediction: 0 (NEGATIVE)
   • Original confidence for positive: 0.0001
   • New prediction: 0 (NEGATIVE)
   • New confidence for positive: 0.0746
   • Change in positive confidence: +0.0745
   • Prediction flip success: ❌ NO

📚 TRAINING DATA IMPACT:
   • Number of examples perturbed: 200
   • Total training examples: 1600
   • Percentage perturbed: 12.50%

🎭 MODEL PERFORMANCE:
   • Original model test accuracy: 100.00%
   • Perturbed model test accuracy: 100.00%
   • Performance change: +0.00 percentage points

🧠 KEY INSIGHTS:
   • The perturbation approach shows promise but didn't achieve
 